In [1]:
from spekk import ops
from math import prod

ops.backend.set_backend("numpy")


def create_array(shape, dims) -> ops.array:
    "Create a new array with the given shape and dims."
    x = ops.arange(prod(shape))
    return ops.reshape(x, shape, dims)

# Indexing with Named Dimensions

Indexing is essential for selecting subsets of data from arrays. However, [NumPy's indexing rules](https://numpy.org/doc/stable/user/basics.indexing.html#advanced-indexing) can be complex, especially when arrays have different shapes that need to be combined.

**Broadcasting** is the automatic alignment of arrays with different shapes during operations. In traditional NumPy, this happens by position - dimensions are matched from right to left. With named dimensions, broadcasting becomes intuitive: dimensions are matched by name, making the intent clear and reducing errors.

**Key concept**: When you index an array using another array, the indexed dimension is replaced by the dimensions of the indexing array.

This enables powerful data selection patterns while maintaining dimension names for clarity.


### Example 1

Select specific elements from a dimension using an array of indices. The indexed dimension (`height`) is replaced by the dimensions of the indexing array (`height_samples`).


In [2]:
# Select height indices [0, 2, 0, 1, 2]
x = create_array((2, 3, 4), ["batch", "height", "width"])
indices = ops.array([0, 2, 0, 1, 2], dims=["height_samples"])
result = x[{"height": indices}]

# "height" (size 3) is replaced by "height_samples" (size 5)
assert result.dim_sizes == {"batch": 2, "height_samples": 5, "width": 4}

**Special case**: When the indexing array uses the same dimension name, the dimension size can change but the name stays the same.


In [3]:
# Reorder height: [2, 0, 1] using same dimension name
x = create_array((2, 3, 4), ["batch", "height", "width"])
indices = ops.array([0, 2, 0, 1, 2], dims=["height"])  # same name
result = x[{"height": indices}]

# "height" keeps its name but changes from size 3 to size 5 (reordered)
assert result.dim_sizes == {"batch": 2, "height": 5, "width": 4}

### Example 2: Multi-dimensional indexing arrays

Indexing arrays can have multiple dimensions.

**Key rules:**

- New dimensions in the indexing array are added to the result
- Shared dimensions must have matching sizes (after any slicing)


In [4]:
# Indexing array introduces a new dimension "channels"
x = create_array((2, 3, 4), ["batch", "height", "width"])
indices = ops.zeros((5, 6), dims=["height_samples", "channels"], dtype="int32")
result = x[{"height": indices}]

# "height" → "height_samples" + "channels" gets added
assert result.dim_sizes == {"batch": 2, "height_samples": 5, "channels": 6, "width": 4}

In [5]:
# Indexing array shares a dimension with the target array
x = create_array((2, 3, 4), ["batch", "height", "width"])
indices = ops.zeros((5, 4), dims=["height_samples", "width"], dtype="int32")
result = x[{"height": indices}]

# "width" sizes must match (both are 4) ✓
assert result.dim_sizes == {"batch": 2, "height_samples": 5, "width": 4}

In [6]:
import pytest

# Error: dimension sizes don't match
x = create_array((2, 3, 4), ["batch", "height", "width"])
indices = ops.zeros((5, 2), dims=["samples", "width"], dtype="int32")
# "width" is size 4 in x, but size 2 in indices ✗
with pytest.raises(ValueError):
    result = x[{"height": indices}]

In [7]:
# Fix: slice "width" to match indexing array size
x = create_array((2, 3, 4), ["batch", "height", "width"])
indices = ops.zeros((5, 2), dims=["samples", "width"], dtype="int32")

# Slice makes "width" size 2, now it matches indices ✓
result = x[{"height": indices, "width": slice(None, None, 2)}]
assert result.dim_sizes == {"batch": 2, "samples": 5, "width": 2}

### Two indexing syntaxes

**Dictionary syntax** (explicit): `x[{"dim": index}]`  
**Tuple syntax** (concise): `x["dim", index, "dim2", index2]`

Both are equivalent - use whichever feels more natural.


In [8]:
x = create_array((2, 3, 4), ["batch", "height", "width"])
indices = ops.zeros((5, 2), dims=["samples", "width"], dtype="int32")

# These are identical:
result1 = x[{"height": indices, "width": slice(None, None, 2)}]
result2 = x["height", indices, "width", ::2]

assert result1.dim_sizes == result2.dim_sizes == {"batch": 2, "samples": 5, "width": 2}
assert ops.all(result1 == result2)